# Protobuf Basics — 03: Protobuf Serialization

Serializing Python objects to Protobuf binary format and back.
This is how gRPC services and Kafka producers create Protobuf messages.

Topics: protobuf Python library, serialize to bytes, deserialize from bytes,
binary format inspection, size comparison with JSON/Avro.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:35:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Install and Verify protobuf Library



In [2]:

import subprocess
result=subprocess.run(["pip3","install","--quiet","--break-system-packages","protobuf>=4.21"],capture_output=True,text=True)
try:
    import google.protobuf
    print(f"protobuf {google.protobuf.__version__} ✅")
except ImportError:
    print("protobuf not installed — will use manual binary simulation")


protobuf not installed — will use manual binary simulation


## Step 2 — Manual Protobuf Wire Format

Understanding the binary format without needing protoc.

In [3]:

import struct

def encode_varint(value):
    """Encode integer as protobuf varint."""
    result = b""
    while value > 0x7F:
        result += bytes([(value & 0x7F) | 0x80])
        value >>= 7
    result += bytes([value & 0x7F])
    return result

def encode_field(field_number, wire_type, value):
    """Encode a single protobuf field: (field_number << 3) | wire_type."""
    tag = (field_number << 3) | wire_type
    return encode_varint(tag) + value

def encode_string(field_number, s):
    """Wire type 2 = length-delimited (string, bytes, embedded messages)."""
    encoded = s.encode('utf-8')
    return encode_field(field_number, 2, encode_varint(len(encoded)) + encoded)

def encode_int64(field_number, n):
    """Wire type 0 = varint."""
    return encode_field(field_number, 0, encode_varint(n))

def encode_double(field_number, d):
    """Wire type 1 = 64-bit fixed."""
    return encode_field(field_number, 1, struct.pack('<d', d))

# Manually encode an Order message
def encode_order(order_id, customer_id, product, category, price, quantity, status):
    msg = b""
    msg += encode_int64(1, order_id)
    msg += encode_int64(2, customer_id)
    msg += encode_string(3, product)
    msg += encode_string(4, category)
    msg += encode_double(5, price)
    msg += encode_int64(6, quantity)
    msg += encode_string(7, status)
    return msg

# Encode a sample order
order_bytes = encode_order(1, 101, "Laptop Pro", "Electronics", 999.99, 2, "confirmed")
import json as pyjson
order_json  = pyjson.dumps({"order_id":1,"customer_id":101,"product":"Laptop Pro",
                             "category":"Electronics","price":999.99,"quantity":2,"status":"confirmed"})

print(f"Same order, different formats:")
print(f"  Protobuf : {len(order_bytes):>4} bytes  {order_bytes.hex()[:40]}...")
print(f"  JSON     : {len(order_json):>4} bytes  {order_json[:40]}...")
print(f"  Ratio    : Protobuf is {len(order_json)/len(order_bytes):.1f}x smaller")


Same order, different formats:
  Protobuf :   51 bytes  080110651a0a4c6170746f702050726f220b456c...
  JSON     :  142 bytes  {"order_id": 1, "customer_id": 101, "pro...
  Ratio    : Protobuf is 2.8x smaller


## Step 3 — Serialize/Deserialize with Python protobuf



In [4]:

import pathlib, subprocess

PROTO_DIR = f"{DATA_DIR}/protos"
pathlib.Path(PROTO_DIR).mkdir(exist_ok=True)

proto_content = """syntax = "proto3";
package example;
message Order {
  int64  order_id    = 1;
  int64  customer_id = 2;
  string product     = 3;
  string category    = 4;
  double price       = 5;
  int32  quantity    = 6;
  double revenue     = 7;
  string status      = 8;
}
"""
pathlib.Path(f"{PROTO_DIR}/orders.proto").write_text(proto_content)

# Try to compile with grpcio-tools (includes protoc)
result = subprocess.run(
    ["python3","-m","grpc_tools.protoc",f"--proto_path={PROTO_DIR}",
     f"--python_out={PROTO_DIR}",f"--descriptor_set_out={PROTO_DIR}/orders.desc",
     "--include_imports",f"{PROTO_DIR}/orders.proto"],
    capture_output=True, text=True
)
if result.returncode != 0:
    result = subprocess.run(
        ["pip3","install","--quiet","--break-system-packages","grpcio-tools"],
        capture_output=True, text=True
    )
    result = subprocess.run(
        ["python3","-m","grpc_tools.protoc",f"--proto_path={PROTO_DIR}",
         f"--python_out={PROTO_DIR}",f"--descriptor_set_out={PROTO_DIR}/orders.desc",
         "--include_imports",f"{PROTO_DIR}/orders.proto"],
        capture_output=True, text=True
    )

if result.returncode == 0 and pathlib.Path(f"{PROTO_DIR}/orders_pb2.py").exists():
    import sys; sys.path.insert(0, PROTO_DIR)
    import importlib.util
    spec = importlib.util.spec_from_file_location("orders_pb2", f"{PROTO_DIR}/orders_pb2.py")
    orders_pb2 = importlib.util.module_from_spec(spec); spec.loader.exec_module(orders_pb2)

    order = orders_pb2.Order()
    order.order_id=1; order.customer_id=101; order.product="Laptop Pro"
    order.category="Electronics"; order.price=999.99; order.quantity=2
    order.revenue=1999.98; order.status="confirmed"

    serialized = order.SerializeToString()
    print(f"Serialized: {len(serialized)} bytes")
    print(f"Hex: {serialized.hex()}")

    order2 = orders_pb2.Order()
    order2.ParseFromString(serialized)
    print(f"\nDeserialized: order_id={order2.order_id} product={order2.product} price={order2.price}")
    print("✅ Round-trip serialization successful")
else:
    print("grpcio-tools not available — using manual encoding from Step 2")
    print("Manual encoded bytes:", order_bytes.hex())
    print(f"Size: {len(order_bytes)} bytes")


grpcio-tools not available — using manual encoding from Step 2
Manual encoded bytes: 080110651a0a4c6170746f702050726f220b456c656374726f6e6963732952b81e85eb3f8f4030023a09636f6e6669726d6564
Size: 51 bytes


## Step 4 — Batch Serialization Performance



In [5]:

import time, json as pyjson, random

random.seed(42)
N = 10_000
CATS=["Electronics","Books","Clothing","Food","Sports"]
CTRS=["US","UK","DE","FR","JP"]

orders = [{"order_id":i,"customer_id":random.randint(1,1000),
           "product":f"P_{random.randint(1,50)}","category":random.choice(CATS),
           "price":round(random.uniform(5,500),2),"quantity":random.randint(1,10),
           "revenue":0.0,"status":"confirmed"} for i in range(N)]
for o in orders: o["revenue"]=round(o["price"]*o["quantity"],2)

# JSON serialization speed
t0=time.time()
json_bytes = [pyjson.dumps(o).encode() for o in orders]
t_json = time.time()-t0
json_total = sum(len(b) for b in json_bytes)

# Manual Protobuf
t0=time.time()
proto_bytes_list = [encode_order(o["order_id"],o["customer_id"],o["product"],
                                  o["category"],o["price"],o["quantity"],o["status"])
                    for o in orders]
t_proto = time.time()-t0
proto_total = sum(len(b) for b in proto_bytes_list)

print(f"Batch serialization of {N:,} orders:")
print(f"  JSON    : {t_json:.3f}s  {json_total/1024:.0f} KB  avg {json_total/N:.0f} bytes/msg")
print(f"  Protobuf: {t_proto:.3f}s  {proto_total/1024:.0f} KB  avg {proto_total/N:.0f} bytes/msg")
print(f"  Size ratio: {json_total/proto_total:.1f}x smaller with Protobuf")


Batch serialization of 10,000 orders:
  JSON    : 0.039s  1498 KB  avg 153 bytes/msg
  Protobuf: 0.060s  415 KB  avg 42 bytes/msg
  Size ratio: 3.6x smaller with Protobuf


## Summary



In [6]:

print("""
Protobuf serialization API (Python):
  # Serialize
  msg = MyMessage()
  msg.field_name = value
  binary = msg.SerializeToString()   # → bytes

  # Deserialize
  msg2 = MyMessage()
  msg2.ParseFromString(binary)       # from bytes

  # From Spark binary column (Kafka value)
  from pyspark.sql.protobuf.functions import from_protobuf
  df.select(
      from_protobuf(col("value"), "Order", "path/to/orders.desc").alias("order")
  ).select("order.*")

Wire format summary:
  varint (field 1-15): 2 bytes overhead per field
  int value 0-127: 1 byte (varint compression!)
  string "hello": 1 tag byte + 1 length byte + 5 data bytes = 7 bytes
  JSON "status":"confirmed" = 19 bytes vs Protobuf field 7 "confirmed" = ~12 bytes
""")



Protobuf serialization API (Python):
  # Serialize
  msg = MyMessage()
  msg.field_name = value
  binary = msg.SerializeToString()   # → bytes

  # Deserialize
  msg2 = MyMessage()
  msg2.ParseFromString(binary)       # from bytes

  # From Spark binary column (Kafka value)
  from pyspark.sql.protobuf.functions import from_protobuf
  df.select(
      from_protobuf(col("value"), "Order", "path/to/orders.desc").alias("order")
  ).select("order.*")

Wire format summary:
  varint (field 1-15): 2 bytes overhead per field
  int value 0-127: 1 byte (varint compression!)
  string "hello": 1 tag byte + 1 length byte + 5 data bytes = 7 bytes
  JSON "status":"confirmed" = 19 bytes vs Protobuf field 7 "confirmed" = ~12 bytes

